# Multi-View Hierarchical Classification - Inference & Retrieval

This notebook implements the unified hierarchical classification system with:
- **Multi-view scoring**: score(n) = max(s_label, s_def, s_ex, s_emp)
- **Global retrieval**: Build candidate set C from all views
- **Top-down traversal**: Beam search with stopping logic (avoid over-specification)
- **Bottom-up validation**: Scoped to C only (avoid wrong-branch errors)
- **Short-query rule**: Prefer label/evidence views for ≤2 token queries

## System Architecture

```
Input Query q
    ↓
[Global Retrieval] → Candidate Set C (top-K nodes from all levels)
    ↓
[Top-Down Beam Traversal with Stopping]
    - Start at root level
    - For each level: expand top-k candidates within C
    - Stop when confidence plateau detected
    ↓
[Bottom-Up Validation] (scoped to C)
    - Validate leaf → parent chain consistency
    - Only consider ancestors within C
    ↓
Output: Predicted path with confidence scores
```

In [ ]:
"""Multi-View Hierarchical Classification - Inference & Retrieval"""
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple, Set
from dataclasses import dataclass

import taxomind.utils.taxonomy_utils as taxo_utils
from taxomind.utils import embedding_utils

In [ ]:
# Configuration
TAXONOMY_KEY = "ISCO"  # Can be changed to "ISCO" or other taxonomy keys

# Retrieval parameters
TOP_K_GLOBAL = 60  # Size of candidate set C from global retrieval
BEAM_WIDTH = 5  # Beam width for top-down traversal
TAU = 10  # Temperature parameter for effective evidence embedding (β blending)
SHORT_QUERY_TOKENS = 2  # Threshold for short-query rule

In [ ]:
# Load Kedro context
%load_ext kedro.ipython
%reload_kedro

## Step 1: Load Embedded Taxonomy

In [ ]:
from taxomind.utils.taxonomy_utils import get_partition_by_key

print(f"Loading embedded taxonomy: {TAXONOMY_KEY}")

# Load partitioned dataset (returns dict of callables)
taxonomy_partitions = catalog.load('taxonomy_embedded')

# Get the specific partition
taxonomy = get_partition_by_key(taxonomy_partitions, TAXONOMY_KEY)

print(f"Loaded {len(taxonomy)} nodes")
print(f"\nEmbedding statistics:")
print(f"  E_label: {taxonomy['E_label'].notna().sum()} embeddings")
print(f"  E_def: {taxonomy['E_def'].notna().sum()} embeddings")
print(f"  E_ex: {taxonomy['E_ex'].notna().sum()} embeddings")
print(f"  C_emb_node: {taxonomy['C_emb_node'].notna().sum()} trained centroids")
print(f"  k_emb_node: {(taxonomy['k_emb_node'] > 0).sum()} nodes with training samples")

# Show structure
taxonomy.head(3)

## Step 2: Compute Effective Evidence Embeddings

For nodes with training data, compute effective evidence embedding:

```
β_n = k_emb_node[n] / (k_emb_node[n] + τ)
C_emb_node_eff[n] = normalize((1-β_n)·E_label[n] + β_n·C_emb_node[n])
```

When k_emb_node[n] = 0 (no training data), β_n = 0, so C_emb_node_eff[n] = E_label[n]

In [ ]:
def compute_effective_evidence_embedding(
    E_label: np.ndarray,
    C_emb_node: Optional[np.ndarray],
    k_emb_node: int,
    tau: float = TAU
) -> np.ndarray:
    """
    Compute effective evidence embedding with β blending.
    
    Args:
        E_label: Label embedding (prior)
        C_emb_node: Centroid from training data (evidence)
        k_emb_node: Number of training samples
        tau: Temperature parameter for β calculation
    
    Returns:
        Effective evidence embedding (normalized)
    """
    if k_emb_node == 0 or C_emb_node is None:
        # No training data: use label embedding as-is
        return E_label
    
    # Compute β (evidence weight)
    beta = k_emb_node / (k_emb_node + tau)
    
    # Blend prior and evidence
    blended = (1 - beta) * E_label + beta * C_emb_node
    
    # Normalize
    norm = np.linalg.norm(blended)
    if norm > 0:
        blended = blended / norm
    
    return blended

# Compute effective evidence embeddings for all nodes
def add_effective_embeddings(df: pd.DataFrame, tau: float = TAU) -> pd.DataFrame:
    """Add E_emp (effective evidence) column to taxonomy."""
    df = df.copy()
    
    E_emp_list = []
    for idx, row in df.iterrows():
        E_label = row['E_label']
        C_emb_node = row['C_emb_node']
        k_emb_node = row['k_emb_node']
        
        E_emp = compute_effective_evidence_embedding(
            E_label, C_emb_node, k_emb_node, tau
        )
        E_emp_list.append(E_emp)
    
    df['E_emp'] = E_emp_list
    return df

taxonomy = add_effective_embeddings(taxonomy, tau=TAU)

print(f"✓ Added E_emp column (effective evidence embeddings)")
print(f"  All {len(taxonomy)} nodes have E_emp")
print(f"  {(taxonomy['k_emb_node'] > 0).sum()} nodes use blended evidence")
print(f"  {(taxonomy['k_emb_node'] == 0).sum()} nodes use label prior only")

## Step 3: Multi-View Scoring Function

For each node n and query q:

```
s_label[n] = cos(q, E_label[n])
s_def[n] = cos(q, E_def[n]) if E_def[n] exists else -∞
s_ex[n] = cos(q, E_ex[n]) if E_ex[n] exists else -∞
s_emp[n] = cos(q, E_emp[n])

score(n) = max(s_label, s_def, s_ex, s_emp)
```

**Short-query rule**: For queries with ≤2 tokens, only use label and evidence views:
```
score(n) = max(s_label, s_emp)
```

In [ ]:
@dataclass
class MultiViewScores:
    """Container for multi-view similarity scores."""
    s_label: float
    s_def: float
    s_ex: float
    s_emp: float
    
    def max_score(self, short_query: bool = False) -> float:
        """Get max score across views."""
        if short_query:
            # Short query: only label and evidence
            return max(self.s_label, self.s_emp)
        else:
            # Full query: all views
            return max(self.s_label, self.s_def, self.s_ex, self.s_emp)
    
    def best_view(self, short_query: bool = False) -> str:
        """Return which view gave the best score."""
        if short_query:
            return 'label' if self.s_label >= self.s_emp else 'evidence'
        else:
            scores = {'label': self.s_label, 'def': self.s_def, 
                     'ex': self.s_ex, 'evidence': self.s_emp}
            return max(scores, key=scores.get)


def compute_multiview_scores(
    query_emb: np.ndarray,
    node: pd.Series
) -> MultiViewScores:
    """
    Compute similarity scores for all views of a node.
    
    Args:
        query_emb: Query embedding (normalized)
        node: Taxonomy node row with E_label, E_def, E_ex, E_emp
    
    Returns:
        MultiViewScores with cosine similarities for each view
    """
    # Label view (always present)
    s_label = np.dot(query_emb, node['E_label'])
    
    # Definition view (if present)
    s_def = -np.inf
    if node['E_def'] is not None:
        s_def = np.dot(query_emb, node['E_def'])
    
    # Examples view (if present)
    s_ex = -np.inf
    if node['E_ex'] is not None:
        s_ex = np.dot(query_emb, node['E_ex'])
    
    # Evidence view (always present)
    s_emp = np.dot(query_emb, node['E_emp'])
    
    return MultiViewScores(s_label, s_def, s_ex, s_emp)


# Test with a sample query
test_query = "education"
embedding_model = embedding_utils.load_embedding_model(context.params['model_name'])
query_embeddings, _ = embedding_utils.encode_texts(
    embedding_model,
    [test_query],
    embed_all=True,
    batch_size=1,
    show_progress_bar=False,
)
query_emb = query_embeddings[0]

# Score a sample node
sample_node = taxonomy.iloc[0]
scores = compute_multiview_scores(query_emb, sample_node)

print(f"Sample multi-view scores for '{test_query}':")
print(f"  Node: {sample_node['code']} - {sample_node['label']}")
print(f"  s_label: {scores.s_label:.4f}")
print(f"  s_def: {scores.s_def:.4f}")
print(f"  s_ex: {scores.s_ex:.4f}")
print(f"  s_emp: {scores.s_emp:.4f}")
print(f"  max_score: {scores.max_score():.4f}")
print(f"  best_view: {scores.best_view()}")


## Step 4: Global Retrieval - Build Candidate Set C

Retrieve top-K nodes from the entire taxonomy (all levels) to form candidate set C.
This reduces the search space for top-down traversal and bottom-up validation.

In [ ]:
def global_retrieval(
    query_emb: np.ndarray,
    taxonomy: pd.DataFrame,
    top_k: int = TOP_K_GLOBAL,
    short_query: bool = False
) -> pd.DataFrame:
    """
    Retrieve top-K nodes from entire taxonomy using multi-view scoring.
    
    Args:
        query_emb: Query embedding
        taxonomy: Full taxonomy DataFrame
        top_k: Number of candidates to retrieve
        short_query: Whether to apply short-query rule
    
    Returns:
        DataFrame of top-K candidates with scores
    """
    results = []
    
    for idx, node in taxonomy.iterrows():
        scores = compute_multiview_scores(query_emb, node)
        max_score = scores.max_score(short_query)
        best_view = scores.best_view(short_query)
        
        results.append({
            'code': node['code'],
            'label': node['label'],
            'level': node['level'],
            'parentCode': node['parentCode'],
            'isLeaf': node['isLeaf'],
            'score': max_score,
            'best_view': best_view,
            's_label': scores.s_label,
            's_def': scores.s_def,
            's_ex': scores.s_ex,
            's_emp': scores.s_emp,
        })
    
    candidates = pd.DataFrame(results)
    candidates = candidates.sort_values('score', ascending=False).head(top_k)
    
    return candidates


# Test global retrieval
test_query = "education"
query_tokens = test_query.split()
is_short_query = len(query_tokens) <= SHORT_QUERY_TOKENS

embedding_model = embedding_utils.load_embedding_model(context.params['model_name'])
query_embeddings, _ = embedding_utils.encode_texts(
    embedding_model,
    [test_query],
    embed_all=True,
    batch_size=1,
    show_progress_bar=False,
)
query_emb = query_embeddings[0]
candidates = global_retrieval(query_emb, taxonomy, top_k=TOP_K_GLOBAL, short_query=is_short_query)

# print(f"Global retrieval for: '{test_query}'")
# print(f"  Short query: {is_short_query} (tokens: {len(query_tokens)})")
# print(f"  Retrieved {len(candidates)} candidates")
# print(f"
Top 10 candidates:")


In [ ]:
candidates

## Step 5: Top-Down Beam Traversal with Stopping

Traverse hierarchy level-by-level with beam search:
1. Start at root level (level 1)
2. For each level, expand top-k candidates from beam
3. Only consider children that are in candidate set C
4. Stop when confidence plateau detected (score improvement < threshold)

This avoids over-specification by stopping before forcing a choice in low-confidence regions.

In [ ]:
@dataclass
class BeamNode:
    """Node in beam search."""
    code: str
    label: str
    level: int
    score: float
    best_view: str
    path: List[str]  # Path of codes from root to this node


def get_children_in_candidates(
    parent_code: str,
    candidates: pd.DataFrame,
    taxonomy: pd.DataFrame
) -> pd.DataFrame:
    """
    Get children of a node that are in the candidate set.
    
    Args:
        parent_code: Code of parent node
        candidates: Candidate set C from global retrieval
        taxonomy: Full taxonomy (to look up children)
    
    Returns:
        DataFrame of children that are in candidates
    """
    # Get all children from taxonomy
    children = taxonomy[taxonomy['parentCode'] == parent_code]
    
    # Filter to only those in candidate set
    candidate_codes = set(candidates['code'])
    children_in_C = children[children['code'].isin(candidate_codes)]
    
    # Merge with candidate scores
    children_with_scores = children_in_C.merge(
        candidates[['code', 'score', 'best_view']],
        on='code',
        how='left'
    )
    
    return children_with_scores


def topdown_beam_search(
    candidates: pd.DataFrame,
    taxonomy: pd.DataFrame,
    beam_width: int = BEAM_WIDTH,
    plateau_threshold: float = 0.05
) -> Tuple[List[BeamNode], bool]:
    """
    Top-down beam search with stopping logic.
    
    Args:
        candidates: Candidate set C from global retrieval
        taxonomy: Full taxonomy DataFrame
        beam_width: Number of candidates to expand at each level
        plateau_threshold: Stop if score improvement < this threshold
    
    Returns:
        (path, stopped_early): List of BeamNodes and whether search stopped early
    """
    # Start with root-level nodes in candidate set
    root_candidates = candidates[candidates['parentCode'] == '__root__']
    
    if len(root_candidates) == 0:
        print("Warning: No root nodes in candidate set")
        return [], False
    
    # Initialize beam with top root candidate
    top_root = root_candidates.iloc[0]
    beam = [BeamNode(
        code=top_root['code'],
        label=top_root['label'],
        level=top_root['level'],
        score=top_root['score'],
        best_view=top_root['best_view'],
        path=[top_root['code']]
    )]
    
    path = [beam[0]]
    prev_best_score = beam[0].score
    stopped_early = False
    
    # Traverse levels
    max_level = candidates['level'].max()
    current_level = 1
    
    while current_level < max_level:
        # Expand beam: get children of current beam nodes
        next_beam_candidates = []
        
        for beam_node in beam:
            children = get_children_in_candidates(
                beam_node.code, candidates, taxonomy
            )
            
            for _, child in children.iterrows():
                next_beam_candidates.append(BeamNode(
                    code=child['code'],
                    label=child['label'],
                    level=child['level'],
                    score=child['score'],
                    best_view=child['best_view'],
                    path=beam_node.path + [child['code']]
                ))
        
        if len(next_beam_candidates) == 0:
            # No children in candidate set - stop here
            stopped_early = True
            break
        
        # Sort and select top-k for new beam
        next_beam_candidates.sort(key=lambda x: x.score, reverse=True)
        beam = next_beam_candidates[:beam_width]
        
        # Check for plateau (stopping condition)
        best_score = beam[0].score
        score_improvement = best_score - prev_best_score
        
        if score_improvement < plateau_threshold:
            # Confidence plateau detected - stop here
            stopped_early = True
            break
        
        # Continue to next level
        path.append(beam[0])
        prev_best_score = best_score
        current_level += 1
    
    return path, stopped_early


# Test top-down beam search
path, stopped = topdown_beam_search(candidates, taxonomy, beam_width=BEAM_WIDTH)

print(f"\nTop-down beam search results:")
print(f"  Path length: {len(path)}")
print(f"  Stopped early: {stopped}")
print(f"\nPredicted path:")
for node in path:
    print(f"  Level {node.level}: {node.code} - {node.label}")
    print(f"    Score: {node.score:.4f} (via {node.best_view})")

## Step 6: Bottom-Up Validation (Scoped to C)

Validate path consistency by checking leaf → parent chain:
1. Start with leaf candidates from C
2. For each leaf, trace parent chain upward
3. Only consider ancestors that are in C
4. Score paths by summing ancestor scores

This provides an alternative path that may correct wrong-branch errors from top-down.

In [ ]:
def get_ancestor_chain_in_candidates(
    node_code: str,
    candidates: pd.DataFrame,
    taxonomy: pd.DataFrame
) -> List[pd.Series]:
    """
    Get ancestor chain from node to root, filtered by candidate set.
    
    Args:
        node_code: Starting node code
        candidates: Candidate set C
        taxonomy: Full taxonomy
    
    Returns:
        List of ancestor nodes (from root to node) that are in C
    """
    candidate_codes = set(candidates['code'])
    chain = []
    
    current_code = node_code
    visited = set()
    
    while current_code and current_code != '__root__':
        if current_code in visited:
            # Cycle detected
            break
        visited.add(current_code)
        
        # Get node from taxonomy
        node_rows = taxonomy[taxonomy['code'] == current_code]
        if len(node_rows) == 0:
            break
        
        node = node_rows.iloc[0]
        
        # Only include if in candidate set
        if current_code in candidate_codes:
            # Get score from candidates
            cand_row = candidates[candidates['code'] == current_code]
            if len(cand_row) > 0:
                node_with_score = node.copy()
                node_with_score['score'] = cand_row.iloc[0]['score']
                node_with_score['best_view'] = cand_row.iloc[0]['best_view']
                chain.append(node_with_score)
        
        current_code = node['parentCode']
    
    # Reverse to get root → leaf order
    return list(reversed(chain))


def bottomup_validation(
    candidates: pd.DataFrame,
    taxonomy: pd.DataFrame,
    top_k: int = 5
) -> List[Tuple[List[BeamNode], float]]:
    """
    Bottom-up validation: score complete paths from leaf to root.
    
    Args:
        candidates: Candidate set C
        taxonomy: Full taxonomy
        top_k: Number of top paths to return
    
    Returns:
        List of (path, total_score) tuples, sorted by score
    """
    # Get leaf candidates
    leaf_candidates = candidates[candidates['isLeaf'] == 1]
    
    paths_with_scores = []
    
    for _, leaf in leaf_candidates.iterrows():
        # Get ancestor chain in C
        chain = get_ancestor_chain_in_candidates(
            leaf['code'], candidates, taxonomy
        )
        
        if len(chain) == 0:
            continue
        
        # Convert to BeamNodes
        path = []
        codes = []
        total_score = 0
        
        for node in chain:
            codes.append(node['code'])
            beam_node = BeamNode(
                code=node['code'],
                label=node['label'],
                level=node['level'],
                score=node['score'],
                best_view=node['best_view'],
                path=codes.copy()
            )
            path.append(beam_node)
            total_score += node['score']
        
        paths_with_scores.append((path, total_score))
    
    # Sort by total score
    paths_with_scores.sort(key=lambda x: x[1], reverse=True)
    
    return paths_with_scores[:top_k]


# Test bottom-up validation
bottomup_paths = bottomup_validation(candidates, taxonomy, top_k=3)

print(f"\nBottom-up validation results:")
print(f"  Found {len(bottomup_paths)} complete paths\n")

for i, (path, total_score) in enumerate(bottomup_paths, 1):
    print(f"Path {i} (total score: {total_score:.4f}):")
    for node in path:
        print(f"  Level {node.level}: {node.code} - {node.label}")
        print(f"    Score: {node.score:.4f} (via {node.best_view})")
    print()

## Step 7: Complete Classification Pipeline

Combine all steps into a complete classification function.

In [ ]:
@dataclass
class ClassificationResult:
    """Result of hierarchical classification."""
    query: str
    is_short_query: bool
    topdown_path: List[BeamNode]
    topdown_stopped_early: bool
    bottomup_paths: List[Tuple[List[BeamNode], float]]
    recommended_path: List[BeamNode]
    candidate_set_size: int
    
    def print_summary(self):
        """Print human-readable summary."""
        print(f"
{'='*60}")
        print(f"CLASSIFICATION RESULT")
        print(f"{'='*60}")
        print(f"Query: '{self.query}'")
        print(f"Short query: {self.is_short_query}")
        print(f"Candidate set size: {self.candidate_set_size}")
        print(f"
RECOMMENDED PATH:")
        for node in self.recommended_path:
            print(f"  Level {node.level}: {node.code} - {node.label}")
            print(f"    Score: {node.score:.4f} (via {node.best_view})")
        print(f"
TOP-DOWN PATH (stopped early: {self.topdown_stopped_early}):")
        for node in self.topdown_path:
            print(f"  Level {node.level}: {node.code} - {node.label}")
            print(f"    Score: {node.score:.4f} (via {node.best_view})")
        print(f"
BOTTOM-UP VALIDATION (top {len(self.bottomup_paths)} paths):")
        for i, (path, score) in enumerate(self.bottomup_paths, 1):
            print(f"  Path {i} (total: {score:.4f}): {' > '.join([n.code for n in path])}")
        print(f"{'='*60}
")


def classify_query(
    query: str,
    taxonomy: pd.DataFrame,
    model_name: str,
    top_k_global: int = TOP_K_GLOBAL,
    beam_width: int = BEAM_WIDTH,
    short_query_threshold: int = SHORT_QUERY_TOKENS
) -> ClassificationResult:
    """
    Complete hierarchical classification pipeline.
    
    Args:
        query: Text query to classify
        taxonomy: Embedded taxonomy DataFrame
        model_name: Embedding model name
        top_k_global: Size of candidate set C
        beam_width: Beam width for top-down search
        short_query_threshold: Token threshold for short-query rule
    
    Returns:
        ClassificationResult with paths and scores
    """
    # Check if short query
    tokens = query.split()
    is_short_query = len(tokens) <= short_query_threshold
    
    # Step 1: Embed query
    embedding_model = embedding_utils.load_embedding_model(model_name)
    query_embeddings, _ = embedding_utils.encode_texts(
        embedding_model,
        [query],
        embed_all=True,
        batch_size=1,
        show_progress_bar=False,
    )
    query_emb = query_embeddings[0]
    
    # Step 2: Global retrieval → candidate set C
    candidates = global_retrieval(
        query_emb, taxonomy, top_k=top_k_global, short_query=is_short_query
    )
    
    # Step 3: Top-down beam search
    topdown_path, stopped_early = topdown_beam_search(
        candidates, taxonomy, beam_width=beam_width
    )
    
    # Step 4: Bottom-up validation
    bottomup_paths = bottomup_validation(candidates, taxonomy, top_k=3)
    
    # Step 5: Recommend best path
    # Use top-down path by default, but could implement logic to prefer bottom-up
    # if top-down stopped very early or has low confidence
    recommended = topdown_path
    
    # If top-down is very short and bottom-up has high-scoring complete paths,
    # prefer bottom-up
    if len(topdown_path) <= 2 and len(bottomup_paths) > 0:
        best_bottomup_path, best_bottomup_score = bottomup_paths[0]
        if len(best_bottomup_path) > len(topdown_path):
            recommended = best_bottomup_path
    
    return ClassificationResult(
        query=query,
        is_short_query=is_short_query,
        topdown_path=topdown_path,
        topdown_stopped_early=stopped_early,
        bottomup_paths=bottomup_paths,
        recommended_path=recommended,
        candidate_set_size=len(candidates)
    )


# Test complete pipeline
test_queries = [
    "education",
    "software developer",
    "farmer growing wheat",
    "nurse",  # Short query
    "restaurant manager responsible for hiring staff and managing budgets"
]

for query in test_queries:
    result = classify_query(
        query, taxonomy, context.params['model_name'],
        top_k_global=TOP_K_GLOBAL,
        beam_width=BEAM_WIDTH
    )
    result.print_summary()


## Step 8: Batch Classification & Evaluation

For evaluating on labeled test data.

In [ ]:
def batch_classify(
    queries: List[str],
    taxonomy: pd.DataFrame,
    model_name: str,
    **classify_kwargs
) -> List[ClassificationResult]:
    """
    Classify multiple queries.
    
    Args:
        queries: List of text queries
        taxonomy: Embedded taxonomy
        model_name: Embedding model name
        **classify_kwargs: Additional args for classify_query
    
    Returns:
        List of ClassificationResults
    """
    results = []
    
    for i, query in enumerate(queries):
        print(f"Classifying {i+1}/{len(queries)}: {query[:50]}...")
        result = classify_query(query, taxonomy, model_name, **classify_kwargs)
        results.append(result)
    
    return results


# Example: batch classify test queries
test_batch = [
    "software developer",
    "farmer",
    "doctor",
    "teacher"
]

batch_results = batch_classify(
    test_batch,
    taxonomy,
    context.params['model_name'],
    top_k_global=TOP_K_GLOBAL,
    beam_width=BEAM_WIDTH
)

print(f"\nBatch classification complete: {len(batch_results)} queries processed")

## Next Steps

1. **Incremental Learning**: Implement training data ingestion to update C_emb_node and k_emb_node
2. **Evaluation Metrics**: Implement hierarchical precision/recall at different levels
3. **Hyperparameter Tuning**: Optimize τ, beam_width, plateau_threshold, top_k_global
4. **Integration**: Create Kedro pipeline for inference
5. **API**: Expose classification as REST endpoint